In [3]:
import matplotlib.pyplot as plt
from datetime import datetime
import re
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np

tpcs = [i for i in range(1,12)]  # Added run19553
run_number = 19655

plot_colors = ['tab:blue']  # Unique color for each run
subfile_colors = [
    'tab:red', 'tab:purple', 'tab:brown', 'tab:orange',
    'tab:green', 'tab:pink', 'tab:gray', 'tab:olive',
    'tab:cyan', 'tab:blue', 'tab:orange'
]

dma_ylim = 350

pdf_filename = 'run{}_all_tpc_plots_{}.pdf'.format(run_number, datetime.now().strftime('%Y%m%d_%H%M%S'))

with PdfPages(pdf_filename) as pdf:

    for tpc in tpcs:

        print(f'Processing TPC {tpc} (PDF)')
        # Repeat the parsing and plotting logic for each TPC
        # (reuse your parsing code above, but do not call plt.show())
        # Read raw data from txt file
        with open(f'/nevis/riverside/data/sc5303/sbnd/continuous_readout/dma_logs/dma_timeout_log_run{run_number}_tpc{tpc:02d}.txt', 'r') as f:
            data = f.read()
        with open(f'/nevis/riverside/data/sc5303/sbnd/continuous_readout/dma_logs/run0{run_number}_sbnd-tpc{tpc:02d}_subfile_index.txt', 'r') as f_sub:
            subfile_data = f_sub.readlines()
        subfile_times = []
        subfile_numbers = []

        ls_re = re.compile(r"(?P<month>\w{3})\s+(?P<day>\d{1,2})\s+(?P<time>\d{1,2}:\d{2})\s+(?P<filename>\S+)$")


        for entry in subfile_data:
            line = entry.rstrip('\n')
            m = ls_re.search(line)
            filename = None
            month_str = None
            day_str = None
            time_str = '00:00'
            if m:
                month_str = m.group('month')
                day_str = m.group('day')
                time_str = m.group('time')
                filename = m.group('filename')
            else:
                tokens = line.split()
                if not tokens:
                    continue
                filename = tokens[-1]
                time_token = None
                for t in reversed(tokens[:-1]):
                    if re.match(r"^\d{1,2}:\d{2}$", t):
                        time_token = t
                        break
                time_str = time_token or '00:00'
                try:
                    time_index = tokens.index(time_str)
                    if time_index >= 2:
                        month_str = tokens[time_index - 2]
                        day_str = tokens[time_index - 1]
                except ValueError:
                    pass
            year = None
            month_file = None
            day_file = None
            if month_str and day_str:
                try:
                    year = datetime.now().year
                    month_file = datetime.strptime(month_str, '%b').month
                    day_file = int(day_str)
                except Exception:
                    year = None
                    month_file = None
                    day_file = None
            if year is None and filename:
                date_match = re.search(r"(\d{4})\.(\d{1,2})\.(\d{1,2})", filename)
                if date_match:
                    year = int(date_match.group(1))
                    month_file = int(date_match.group(2))
                    day_file = int(date_match.group(3))
            if year is None:
                continue
            try:
                hour, minute = map(int, time_str.split(':'))
            except Exception:
                hour, minute = 0, 0
            try:
                dt = datetime(year, month_file, day_file, hour, minute)
                subfile_times.append(dt)
            except Exception:
                continue
            subfile_num = None
            if filename:
                num_match = re.search(r'_subfile(\d+)_', filename)
                if not num_match:
                    num_match = re.search(r'subfile(\d+)_', filename)
                if not num_match:
                    num_match = re.search(r'_(\d{1,4})_TPC', filename)
                if num_match:
                    try:
                        subfile_num = int(num_match.group(1))
                    except Exception:
                        subfile_num = None
            if subfile_num is not None:
                subfile_numbers.append(subfile_num)
        lines = [line for line in data.strip().split('\n') if line]
        times = []
        counts = []
        run_number_str = None
        for line in lines:
            parts = [p.strip() for p in line.split('|')]
            if len(parts) < 3:
                continue
            time_str = parts[0]
            run_str = parts[1]
            count_str = parts[2]
            try:
                times.append(datetime.strptime(time_str, "%Y-%m-%d %H:%M:%S"))
            except Exception:
                continue
            try:
                counts.append(int(count_str.split(':')[1].strip()))
            except Exception:
                counts.append(0)
            if run_number_str is None:
                tokens = run_str.split()
                if len(tokens) >= 2:
                    run_number_str = tokens[1]
        fig, ax1 = plt.subplots(figsize=(10, 5))
        if times and counts:
            ax1.plot(times, counts, marker='o', color='tab:blue')
        ax1.set_xlabel('Time')
        ax1.set_ylabel('DMA timed out count', color='tab:blue')
        ax1.tick_params(axis='y', labelcolor='tab:blue')
        ax1.set_title(f'Run {run_number} TPC-{tpc:02d}')
        ax2 = ax1.twinx()
        if subfile_times and subfile_numbers:
            ax2.scatter(subfile_times, subfile_numbers, marker='x', color='tab:red')
        ax2.set_ylabel('Subfile number', color='tab:red')
        ax2.tick_params(axis='y', labelcolor='tab:red')

        ax1.set_ylim([0,6000])
        ax2.set_ylim([0,200])

        plt.xticks(rotation=45)
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
print(f'All plots saved to {pdf_filename}')

Processing TPC 1 (PDF)


Processing TPC 2 (PDF)
Processing TPC 3 (PDF)
Processing TPC 4 (PDF)
Processing TPC 5 (PDF)
Processing TPC 6 (PDF)
Processing TPC 7 (PDF)
Processing TPC 8 (PDF)
Processing TPC 9 (PDF)
Processing TPC 10 (PDF)
Processing TPC 11 (PDF)
All plots saved to run19655_all_tpc_plots_20251019_152909.pdf


In [10]:
import matplotlib.pyplot as plt
from datetime import datetime
import re
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np

tpcs = [i for i in range(1,12)]  # Added run19553
run_number = 19638

plot_colors = ['tab:blue']  # Unique color for each run
subfile_colors = [
    'tab:red', 'tab:purple', 'tab:brown', 'tab:orange',
    'tab:green', 'tab:pink', 'tab:gray', 'tab:olive',
    'tab:cyan', 'tab:blue', 'tab:orange'
]

dma_ylim = 350


for tpc in tpcs:

    print(f'Processing TPC {tpc} (PDF)')
    # Repeat the parsing and plotting logic for each TPC
    # (reuse your parsing code above, but do not call plt.show())
    # Read raw data from txt file
    with open(f'/nevis/riverside/data/sc5303/sbnd/continuous_readout/dma_logs/dma_timeout_log_run{run_number}_tpc{tpc:02d}.txt', 'r') as f:
        data = f.read()
    with open(f'/nevis/riverside/data/sc5303/sbnd/continuous_readout/dma_logs/run0{run_number}_sbnd-tpc{tpc:02d}_subfile_index.txt', 'r') as f_sub:
        subfile_data = f_sub.readlines()
    subfile_times = []
    subfile_numbers = []

    ls_re = re.compile(r"(?P<month>\w{3})\s+(?P<day>\d{1,2})\s+(?P<time>\d{1,2}:\d{2})\s+(?P<filename>\S+)$")


    for entry in subfile_data:
        line = entry.rstrip('\n')
        m = ls_re.search(line)
        filename = None
        month_str = None
        day_str = None
        time_str = '00:00'
        if m:
            month_str = m.group('month')
            day_str = m.group('day')
            time_str = m.group('time')
            filename = m.group('filename')
        else:
            tokens = line.split()
            if not tokens:
                continue
            filename = tokens[-1]
            time_token = None
            for t in reversed(tokens[:-1]):
                if re.match(r"^\d{1,2}:\d{2}$", t):
                    time_token = t
                    break
            time_str = time_token or '00:00'
            try:
                time_index = tokens.index(time_str)
                if time_index >= 2:
                    month_str = tokens[time_index - 2]
                    day_str = tokens[time_index - 1]
            except ValueError:
                pass
        year = None
        month_file = None
        day_file = None
        if month_str and day_str:
            try:
                year = datetime.now().year
                month_file = datetime.strptime(month_str, '%b').month
                day_file = int(day_str)
            except Exception:
                year = None
                month_file = None
                day_file = None
        if year is None and filename:
            date_match = re.search(r"(\d{4})\.(\d{1,2})\.(\d{1,2})", filename)
            if date_match:
                year = int(date_match.group(1))
                month_file = int(date_match.group(2))
                day_file = int(date_match.group(3))
        if year is None:
            continue
        try:
            hour, minute = map(int, time_str.split(':'))
        except Exception:
            hour, minute = 0, 0
        try:
            dt = datetime(year, month_file, day_file, hour, minute)
            subfile_times.append(dt)
        except Exception:
            continue
        subfile_num = None
        if filename:
            num_match = re.search(r'_subfile(\d+)_', filename)
            if not num_match:
                num_match = re.search(r'subfile(\d+)_', filename)
            if not num_match:
                num_match = re.search(r'_(\d{1,4})_TPC', filename)
            if num_match:
                try:
                    subfile_num = int(num_match.group(1))
                except Exception:
                    subfile_num = None
        if subfile_num is not None:
            subfile_numbers.append(subfile_num)
    lines = [line for line in data.strip().split('\n') if line]
    times = []
    counts = []
    run_number_str = None
    for line in lines:
        parts = [p.strip() for p in line.split('|')]
        if len(parts) < 3:
            continue
        time_str = parts[0]
        run_str = parts[1]
        count_str = parts[2]
        try:
            times.append(datetime.strptime(time_str, "%Y-%m-%d %H:%M:%S"))
        except Exception:
            continue
        try:
            counts.append(int(count_str.split(':')[1].strip()))
        except Exception:
            counts.append(0)
        if run_number_str is None:
            tokens = run_str.split()
            if len(tokens) >= 2:
                run_number_str = tokens[1]
    fig, ax1 = plt.subplots(figsize=(10, 5))
    if times and counts:
        ax1.plot(times, counts, marker='o', color='tab:blue')
    ax1.set_xlabel('Time')
    ax1.set_ylabel('DMA timed out count', color='tab:blue')
    ax1.tick_params(axis='y', labelcolor='tab:blue')
    ax1.set_title(f'Run {run_number} TPC-{tpc:02d}')
    ax2 = ax1.twinx()
    if subfile_times and subfile_numbers:
        ax2.scatter(subfile_times, subfile_numbers, marker='x', color='tab:red')
    ax2.set_ylabel('Subfile number', color='tab:red')
    ax2.tick_params(axis='y', labelcolor='tab:red')

    if len(subfile_times) >= 2:
        # Use only the initial 30 points or all if less
        n_points = min(30, len(subfile_times))
        print(f'n_points for TPC {tpc}: {n_points}')
        x = np.array([(t - subfile_times[0]).total_seconds() for t in subfile_times[20:n_points]])
        y = np.array(subfile_numbers[20:n_points])
        if len(x) > 1:
            slope, intercept = np.polyfit(x, y, 1)
            ax2.text(0.05, 0.95, f'Slope: {slope:.3f} subfile/sec', transform=ax2.transAxes, 
                        fontsize=10, verticalalignment='top', color='tab:red')

    print(f'Slope for TPC {tpc}: {slope:.3f} subfile/sec')
    ax1.set_ylim([0,6000])
    ax2.set_ylim([0,200])

    plt.xticks(rotation=45)
    plt.tight_layout()
    # pdf.savefig(fig)
    plt.close(fig)

Processing TPC 1 (PDF)
n_points for TPC 1: 30
Slope for TPC 1: 0.133 subfile/sec
Processing TPC 2 (PDF)
n_points for TPC 2: 30
Slope for TPC 2: 0.008 subfile/sec
Processing TPC 3 (PDF)
n_points for TPC 3: 30
Slope for TPC 3: 0.308 subfile/sec
Processing TPC 4 (PDF)
n_points for TPC 4: 30
Slope for TPC 4: 0.006 subfile/sec
Processing TPC 5 (PDF)
n_points for TPC 5: 30
Slope for TPC 5: 0.007 subfile/sec
Processing TPC 6 (PDF)
n_points for TPC 6: 30
Slope for TPC 6: 0.367 subfile/sec
Processing TPC 7 (PDF)
n_points for TPC 7: 30
Slope for TPC 7: 0.311 subfile/sec
Processing TPC 8 (PDF)
n_points for TPC 8: 30
Slope for TPC 8: 0.120 subfile/sec
Processing TPC 9 (PDF)
n_points for TPC 9: 30
Slope for TPC 9: 0.007 subfile/sec
Processing TPC 10 (PDF)
n_points for TPC 10: 30
Slope for TPC 10: 0.003 subfile/sec
Processing TPC 11 (PDF)
n_points for TPC 11: 30
Slope for TPC 11: 0.008 subfile/sec


In [9]:
0.314  * 1000

314.0